# AAPC (Amino Acid Pair Composition) Encoding

This notebook encodes protein sequences using Amino Acid Pair Composition.

## What is AAPC?

**AAPC** counts the frequency of **adjacent amino acid pairs** in a sequence.

- For 20 amino acids, there are 20 × 20 = **400 possible pairs**
- Each pair (e.g., "AC", "CG", "GH") has its own feature
- Values represent the **frequency** of each pair in the sequence

## Why AAPC for PTM Prediction?

✓ **Captures local patterns** (dipeptide motifs)
✓ **Compact representation** (400 features vs 620 for BLOSUM)
✓ **Position-independent** (works well for variable patterns)
✓ Example: High "CG" frequency might indicate certain PTM sites

## Encoding Method

For sequence "ACDEFG":
- Pairs: AC, CD, DE, EF, FG (5 pairs total)
- Features: freq(AA)=0, freq(AC)=1/5, freq(AD)=0, ..., freq(YY)=0
- Total features: **400 features** (one per possible pair)

## Configuration

In [1]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_FILE = "../data_engineered/train_with_features.csv"  # From notebook 02
OUTPUT_FILE = "../data_engineered/aapc/train_with_features_aapc.csv"

# Processing
CHUNK_SIZE = 10000  # Process in chunks to save memory

print("Configuration loaded:")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Chunk size: {CHUNK_SIZE}")

Configuration loaded:
  Input: ../data_engineered/train_with_features.csv
  Output: ../data_engineered/aapc/train_with_features_aapc.csv
  Chunk size: 10000


## Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

print("✓ Libraries imported")

✓ Libraries imported


## AAPC Encoding Functions

In [3]:
# Standard 20 amino acids
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def sequence_to_aapc(sequence):
    """
    Encode a sequence using Amino Acid Pair Composition.
    
    Returns:
        1D array of 400 features (20x20 possible pairs)
    """
    # Initialize feature vector for all possible pairs
    aapc_vector = np.zeros(400, dtype=np.float32)
    
    # Count pairs
    if len(sequence) < 2:
        return aapc_vector  # Return zeros if sequence too short
    
    total_pairs = 0
    for i in range(len(sequence) - 1):
        aa1 = sequence[i]
        aa2 = sequence[i + 1]
        
        # Only count valid pairs
        if aa1 in AA_TO_IDX and aa2 in AA_TO_IDX:
            idx1 = AA_TO_IDX[aa1]
            idx2 = AA_TO_IDX[aa2]
            pair_idx = idx1 * 20 + idx2  # Map 2D to 1D index
            aapc_vector[pair_idx] += 1.0
            total_pairs += 1
    
    # Normalize by total number of pairs (convert counts to frequencies)
    if total_pairs > 0:
        aapc_vector /= total_pairs
    
    return aapc_vector

def encode_chunk(chunk_df):
    """
    Encode a chunk of sequences using AAPC.
    
    Returns DataFrame with original features + AAPC features.
    """
    sequences = chunk_df['Sequence'].values
    
    # Encode all sequences in chunk
    aapc_features = np.vstack([
        sequence_to_aapc(seq)
        for seq in sequences
    ])
    
    # Create column names: aapc_AA, aapc_AC, ..., aapc_YY
    aapc_cols = []
    for aa1 in AMINO_ACIDS:
        for aa2 in AMINO_ACIDS:
            aapc_cols.append(f"aapc_{aa1}{aa2}")
    
    # Create DataFrame with AAPC features
    aapc_df = pd.DataFrame(aapc_features, columns=aapc_cols, index=chunk_df.index)
    
    # Combine with original features (drop Sequence to save space)
    result_df = pd.concat([chunk_df.drop(columns=['Sequence']), aapc_df], axis=1)
    
    return result_df

print("✓ AAPC encoding functions defined")
print(f"  Total possible pairs: {len(AMINO_ACIDS) * len(AMINO_ACIDS)}")

✓ AAPC encoding functions defined
  Total possible pairs: 400


## Load and Encode Data

In [4]:
print("="*80)
print("LOADING AND ENCODING DATA")
print("="*80)

# Check if input file exists
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run 02_feature_engineering.ipynb first."
    )

# Get total rows
total_rows = sum(1 for _ in open(INPUT_FILE)) - 1  # -1 for header
print(f"\nTotal sequences to encode: {total_rows:,}")

# Process in chunks
chunks_processed = []
for chunk in tqdm(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE), 
                   total=(total_rows // CHUNK_SIZE) + 1,
                   desc="Encoding chunks"):
    
    encoded_chunk = encode_chunk(chunk)
    chunks_processed.append(encoded_chunk)

# Combine all chunks
print("\nCombining chunks...")
encoded_df = pd.concat(chunks_processed, ignore_index=True)

print(f"\n✓ Encoding complete!")
print(f"  Total samples: {len(encoded_df):,}")
print(f"  Total features: {len(encoded_df.columns)}")
print(f"  AAPC features: 400")
print(f"  Original features: {len(encoded_df.columns) - 400}")

LOADING AND ENCODING DATA

Total sequences to encode: 89,010


Encoding chunks: 100%|█████████████████████████████████████████████████████████| 9/9 [00:02<00:00,  4.21it/s]


Combining chunks...

✓ Encoding complete!
  Total samples: 89,010
  Total features: 438
  AAPC features: 400
  Original features: 38


## Verify Encoding

In [5]:
print("\n" + "="*80)
print("VERIFICATION")
print("="*80)

# Check for NaN/Inf values
nan_count = encoded_df.isnull().sum().sum()
inf_count = np.isinf(encoded_df.select_dtypes(include=[np.number])).sum().sum()

print(f"\nData quality:")
print(f"  NaN values: {nan_count}")
print(f"  Inf values: {inf_count}")

if nan_count > 0 or inf_count > 0:
    print("  ⚠️  Warning: Found invalid values!")
else:
    print("  ✓ All values are valid")

# Check AAPC features
aapc_cols = [col for col in encoded_df.columns if col.startswith('aapc_')]
print(f"\nAAPC features:")
print(f"  Count: {len(aapc_cols)}")
print(f"  Min value: {encoded_df[aapc_cols].min().min():.4f}")
print(f"  Max value: {encoded_df[aapc_cols].max().max():.4f}")
print(f"  Mean value: {encoded_df[aapc_cols].mean().mean():.4f}")
print(f"  (Values are frequencies, should be between 0 and 1)")

# Show most common pairs
print(f"\nMost common pairs (average frequency across all sequences):")
pair_means = encoded_df[aapc_cols].mean().sort_values(ascending=False)
for i, (col, val) in enumerate(pair_means.head(10).items()):
    pair = col.replace('aapc_', '')
    print(f"  {i+1}. {pair}: {val:.4f}")


VERIFICATION

Data quality:
  NaN values: 0
  Inf values: 0
  ✓ All values are valid

AAPC features:
  Count: 400
  Min value: 0.0000
  Max value: 0.5667
  Mean value: 0.0025
  (Values are frequencies, should be between 0 and 1)

Most common pairs (average frequency across all sequences):
  1. LL: 0.0094
  2. SS: 0.0066
  3. LS: 0.0066
  4. SL: 0.0066
  5. AL: 0.0065
  6. LA: 0.0063
  7. CL: 0.0060
  8. LC: 0.0059
  9. VL: 0.0059
  10. LE: 0.0058


## Save Encoded Data

In [6]:
print("\n" + "="*80)
print("SAVING ENCODED DATA")
print("="*80)

# Create output directory
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Save to CSV
encoded_df.to_csv(OUTPUT_FILE, index=False)

file_size = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)  # MB

print(f"\n✓ Saved to: {OUTPUT_FILE}")
print(f"  File size: {file_size:.1f} MB")
print(f"  Samples: {len(encoded_df):,}")
print(f"  Features: {len(encoded_df.columns)}")

print("\n" + "="*80)
print("✓ AAPC ENCODING COMPLETE!")
print("="*80)
print(f"\nNext step: Run 04_train_val_split.ipynb with this file to create train/val splits.")


SAVING ENCODED DATA

✓ Saved to: ../data_engineered/aapc/train_with_features_aapc.csv
  File size: 187.7 MB
  Samples: 89,010
  Features: 438

✓ AAPC ENCODING COMPLETE!

Next step: Run 04_train_val_split.ipynb with this file to create train/val splits.


## Example: How AAPC Works

```python
# Example sequence: "ACDEFG"
# 
# Adjacent pairs:
# - AC (position 0→1)
# - CD (position 1→2)
# - DE (position 2→3)
# - EF (position 3→4)
# - FG (position 4→5)
# 
# Total: 5 pairs
# 
# Encoding:
# aapc_AC = 1/5 = 0.20
# aapc_CD = 1/5 = 0.20
# aapc_DE = 1/5 = 0.20
# aapc_EF = 1/5 = 0.20
# aapc_FG = 1/5 = 0.20
# aapc_AA = 0.00 (not present)
# aapc_AB = 0.00 (B not in sequence)
# ... (all other pairs = 0)
```

### Why This is Useful:

**Local Patterns**: Some PTM sites might be characterized by specific dipeptide patterns
- High CG frequency might indicate S-nitrosylation sites
- High CP frequency might indicate S-glutathionylation sites

**Compact**: 400 features vs 620 for BLOSUM, faster to train

**Robust**: Position-independent, works even if PTM site position varies